In [1]:
!pip install google-cloud-documentai
!pip install vertexai
!pip install spacy
!pip install nltk
!pip install google-api-core
!python -m spacy download en_core_web_sm

  Using cached google_cloud_bigquery-3.29.0-py2.py3-none-any.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.2 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.2 MB 3.4 MB/s eta 0:00:02
   ---------- ----------------------------- 1.6/6.2 MB 3.0 MB/s eta 0:00:02
   --------------- ------------------------ 2.4/6.2 MB 3.3 MB/s eta 0:00:02
   ---------------- ----------------------- 2.6/6.2 MB 3.2 MB/s eta 0:00:02
   -------------------- ------------------- 3.1/6.2 MB 2.8 MB/s eta 0:00:02
   ----------------------- ---------------- 3.7/6.2 MB 2.7 MB/s eta 0:00:01
   ------------------------- -------------- 3.9/6.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------- ----------- 4.5/6.2 MB 2.4 MB/s eta 0:00:01
   -------------------------------- ------- 5.0/6.2 MB 2.4 MB/s eta 0:00:01
   --------------------------------- ------ 5.2/6.2 MB 2.3 MB/s eta 0:00:01
   ----------------

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 2.1 MB/s eta 0:00:06
     ---- ----------------------------------- 1.3/12.8 MB 2.3 MB/s eta 0:00:05
     ----- ---------------------------------- 1.8/12.8 MB 2.3 MB/s eta 0:00:05
     -------- ------------------------------- 2.6/12.8 MB 2.6 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 2.6 MB/s eta 0:00:04
     ------------- -------------------------- 4.2/12.8 MB 2.9 MB/s eta 0:00:04
     -------------- ------------------------- 4.7/12.8 MB 2.9 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 3.1 MB/s eta 0:00:03
     ------------------- -------------------- 6.3/12.8 MB 3.0 MB/s eta 0:00:03
     ---------------------- ----------------- 7.1/12.8 MB 3.1 MB/s eta 0:00:02
     ------------------------ --------------- 7.9/12.8 MB 3.2 MB/s

In [7]:
import logging
import json
from typing import Optional, Dict, Union, List
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
from vertexai.generative_models import GenerativeModel
import vertexai
import os
import re
import nltk

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

class PDFProcessor:
    def __init__(
        self,
        project_id: str = "gcp-smart-capture",
        location: str = "us",
        processor_id: str = "6d88bf439f34e6a5"
    ):
        self.logger = logging.getLogger(__name__)
        try:
            self.client = documentai.DocumentProcessorServiceClient(
                client_options=ClientOptions(
                    api_endpoint=f"{location}-documentai.googleapis.com"
                )
            )
            self.resource_name = self.client.processor_path(
                project_id, location, processor_id
            )
            self.text_preprocessor = TextPreprocessor()
        except Exception as e:
            self.logger.error(f"Failed to initialize PDFProcessor: {str(e)}")
            raise

    def process_pdf(self, file_path: str) -> str:
        try:
            self.logger.info(f"Processing PDF: {file_path}")
            with open(file_path, "rb") as pdf_file:
                content = pdf_file.read()

            raw_document = documentai.RawDocument(
                content=content,
                mime_type="application/pdf"
            )
            request = documentai.ProcessRequest(
                name=self.resource_name,
                raw_document=raw_document
            )
            result = self.client.process_document(request=request)
            self.logger.info("PDF processing completed successfully")
            return result.document.text
        except FileNotFoundError:
            self.logger.error(f"PDF file not found: {file_path}")
            raise
        except Exception as e:
            self.logger.error(f"Error processing PDF: {str(e)}")
            raise

class TextPreprocessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        self.patterns = {
            'multiple_newlines': r'\n+',
            'multiple_spaces': r'\s+'
        }

    def preprocess(self, text: str) -> str:
        try:
            self.logger.info("Starting text preprocessing")
            text = ' '.join(text.split())
            text = self._normalize_whitespace(text)
            self.logger.info("Text preprocessing completed successfully")
            return text.strip()
        except Exception as e:
            self.logger.error(f"Error during text preprocessing: {str(e)}")
            raise

    def _normalize_whitespace(self, text: str) -> str:
        try:
            for pattern in self.patterns.values():
                text = re.sub(pattern, ' ', text)
            return text.strip()
        except Exception as e:
            self.logger.error(f"Error normalizing whitespace: {str(e)}")
            raise

class EventExtractor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        try:
            vertexai.init(project="gcp-smart-capture", location="us-central1")
            self.model = GenerativeModel("gemini-1.5-pro-002")
            self.generation_config = {
                "candidate_count": 1,
                "max_output_tokens": 8192,
                "temperature": 0,
                "top_p": 0.95,
            }
        except Exception as e:
            self.logger.error(f"Failed to initialize EventExtractor: {str(e)}")
            raise

    def extract_events(self, text: str) -> Dict:
        try:
            self.logger.info("Starting event extraction")
            prompt = f"""
            Extract events from the following text and return a JSON with indentation 4:
            1. Event Name
            2. Event Location
            3. Event start date
            4. Event end date
            5. Description
            
            Text: {text}
            """
            
            response = self.model.generate_content(
                prompt,
                generation_config=self.generation_config
            )
            
            # Clean and format the response
            json_str = response.text.strip()
            if not json_str.startswith('{'):
                json_str = '{' + json_str.split('{', 1)[1]
            if not json_str.endswith('}'):
                json_str = json_str.rsplit('}', 1)[0] + '}'
                
            try:
                events = json.loads(json_str)
            except json.JSONDecodeError:
                self.logger.warning("Failed to parse JSON, returning raw response")
                return response.text
                
            formatted_json = json.dumps(events, indent=4)
            
            self.logger.info("Event extraction completed successfully")
            return formatted_json
        except Exception as e:
            self.logger.error(f"Error during event extraction: {str(e)}")
            raise

class DocumentProcessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        self.pdf_processor = PDFProcessor()
        self.text_preprocessor = TextPreprocessor()
        self.event_extractor = EventExtractor()

    def process_document(self, pdf_path: str) -> str:
        try:
            self.logger.info(f"Starting document processing for: {pdf_path}")
            
            # Extract text from PDF
            raw_text = self.pdf_processor.process_pdf(pdf_path)
            
            # Preprocess text
            processed_text = self.text_preprocessor.preprocess(raw_text)
            
            # Extract events
            events_json = self.event_extractor.extract_events(processed_text)
            
            self.logger.info("Document processing completed successfully")
            return events_json
        except Exception as e:
            self.logger.error(f"Error during document processing: {str(e)}")
            raise


def main():
    logging.info("Starting main application")
    try:
        processor = DocumentProcessor()
        pdf_path = r"C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf"  
        result = processor.process_document(pdf_path)
        print(result)
        logging.info("Application completed successfully")
    except Exception as e:
        logging.error(f"Application failed: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2025-01-24 21:27:08,346 - root - INFO - Starting main application
2025-01-24 21:27:08,360 - __main__ - INFO - Starting document processing for: C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf
2025-01-24 21:27:08,362 - __main__ - INFO - Processing PDF: C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf
2025-01-24 21:27:16,165 - __main__ - INFO - PDF processing completed successfully
2025-01-24 21:27:16,166 - __main__ - INFO - Starting text preprocessing
2025-01-24 21:27:16,168 - __main__ - INFO - Text preprocessing completed successfully
2025-01-24 21:27:16,170 - __main__ - INFO - Starting event extraction
2025-01-24 21:27:42,573 - __main__ - WARNING - Failed to parse JSON, returning raw response
2025-01-24 21:27:42,575 - __main__ - INFO - Document processing completed successfully
2025-01-24 21:27:42,577 - root - INFO - Application completed successfully


```json
[
    {
        "Event Name": "Cloud Summit Hong Kong",
        "Event Location": "Hong Kong",
        "Event start date": "May 23",
        "Event end date": null,
        "Description": null
    },
    {
        "Event Name": "Cloud Summit Paris",
        "Event Location": "Paris",
        "Event start date": "Jun 24",
        "Event end date": null,
        "Description": null
    },
    {
        "Event Name": "Cloud Summit Jakarta",
        "Event Location": "Jakarta",
        "Event start date": "Jun 27",
        "Event end date": null,
        "Description": null
    },
    {
        "Event Name": "Cloud Summit Mumbai",
        "Event Location": "Mumbai",
        "Event start date": "Jul 23",
        "Event end date": null,
        "Description": null
    },
    {
        "Event Name": "Cloud Summit Madrid",
        "Event Location": "Madrid",
        "Event start date": "Jul 28",
        "Event end date": null,
        "Description": null
    },
    {
        "Event Nam